# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token** exactly as the data-contract notebook does: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy` supplies the luck tests, and `scikit-learn` supplies the separation score (AUC), the fold splitters, and a small probe model used only as a leakage detector.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which reuses the data contract's saved results when they exist in the same folder, so running both notebooks in one session scans the warehouse only once.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import nbinom, poisson
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## 1. Build the feature vector

### 1a. The month pairs, rebuilt with the data contract's rules

**Method:** this cell rebuilds the page-month table for February to May 2026 with the same query as the data contract (one row per page per month, position pieces only from days with a valid position), reusing its cached files when they exist. It also reads the client history dates and, from the content table, the keyword fields used as features plus the product and snapshot fields that the leakage hunt needs to test.


In [ ]:
# Identical to the data contract's query, so the cache files are shared between the two notebooks.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"""
    SELECT content_hash_id, keyword_hash_id, content_type, main_intent, search_volume, competition, cpc,
           keyword_token_count, last_optimized_date, optimization_eligible_date, is_deleted, is_published,
           word_count, content_updated_date
    FROM {DIM_CONTENT}""").df()

show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with keyword, product, and snapshot fields"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's eligibility rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair), computes expected clicks for the target and for the baseline, and builds the locked target: clicks in month M+1 at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the baseline: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with expectations and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    for suffix in ("m", "next"):
        rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
        spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
        rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows["expected_next"] = client_aware_expected(rows, "clicks_next", "impressions_next", position_bin(rows["position_next"]))
    rows["expected_m_tier"] = client_aware_expected(rows, "clicks_m", "impressions_m", position_tier(rows["position_m"]))
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    p_next = nbinom.cdf(rows["clicks_next"], 1 / ALPHA, 1 / (1 + ALPHA * rows["expected_next"]))
    rows["target"] = (p_next < 0.05) & (rows["clicks_next"] <= 0.5 * rows["expected_next"])
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

### 1b. The feature vector

Every feature comes from one of two places: the page's own search data in month M, or keyword research done before the page was written. Engineered features:

- **Counts on a log scale** (`log1p`), because impressions range from a handful to millions.
- **Click-through rate in month M**, clicks ÷ impressions.
- **The baseline's expected clicks in month M**, so a model starts from what the transparent rule already knows.
- **Keyword numbers with gaps filled by 0, plus a has-value flag** for each, because the gaps follow content type (feedly articles have none) and the flag keeps the fill honest.
- **Categories one-hot encoded**, with an "unknown" column for missing search intent.

**Method:** this cell defines the feature-building function, applies it to every pair, and checks that every pair yields the same columns, all numeric, with no gaps left.


In [ ]:
# Fixed category lists, read from the content table so every pair gets identical columns.
CONTENT_TYPES = sorted(content["content_type"].dropna().unique())
INTENTS = sorted(content["main_intent"].dropna().unique())


def feature_vector(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the model-ready feature matrix for one pair: month-M search features and keyword-research features only."""
    features = pd.DataFrame(index=rows.index)
    features["log_impressions_m"] = np.log1p(rows["impressions_m"])
    features["log_clicks_m"] = np.log1p(rows["clicks_m"])
    features["ctr_m"] = rows["clicks_m"] / rows["impressions_m"]
    features["position_m"] = rows["position_m"]
    features["position_volatility_m"] = rows["position_volatility_m"]
    features["days_with_impressions_m"] = rows["days_with_impressions_m"]
    features["log_expected_m"] = np.log1p(rows["expected_m_tier"])
    for field in ("search_volume", "competition", "cpc", "main_intent"):
        features[f"has_{field}"] = rows[field].notna().astype(int)
    features["log_search_volume"] = np.log1p(rows["search_volume"]).fillna(0)
    features["competition"] = rows["competition"].fillna(0)
    features["log_cpc"] = np.log1p(rows["cpc"]).fillna(0)
    features["keyword_token_count"] = rows["keyword_token_count"]
    for value in CONTENT_TYPES:
        features[f"content_type={value}"] = (rows["content_type"] == value).astype(int)
    for value in INTENTS:
        features[f"intent={value}"] = (rows["main_intent"] == value).astype(int)
    features["intent=unknown"] = rows["main_intent"].isna().astype(int)
    return features


vectors = {name: feature_vector(rows) for name, rows in pairs.items()}
columns = list(next(iter(vectors.values())).columns)
for name, features in vectors.items():
    assert list(features.columns) == columns, f"{name}: columns differ from the first pair."
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes), f"{name}: a feature is not numeric."
    assert not features.isna().any().any(), f"{name}: a feature still has gaps."

show_table("Feature vector, every pair", pd.DataFrame({
    "Rows": [f"{len(features):,}" for features in vectors.values()],
    "Features": [features.shape[1] for features in vectors.values()],
    "Gaps left": [int(features.isna().sum().sum()) for features in vectors.values()],
}, index=pd.Index(list(vectors), name="Pair")))

## 2. Feature notes (meaning, missing, categorical, available-when?)

**Method:** this cell writes one note per feature (what it means, where it comes from, and when it is known), checks that every column of the feature vector has a note and that no note's source is month M+1, and measures how often each underlying field was missing before any fill, on the first pair.


In [ ]:
# Feature: (meaning, source, when it is known).
MONTH_M = "Month M search data"
KEYWORDS = "Keyword research (content table)"
NOTES = {
    "log_impressions_m": ("Times the page was shown in month M, log scale", MONTH_M, "Last day of month M"),
    "log_clicks_m": ("Clicks earned in month M, log scale", MONTH_M, "Last day of month M"),
    "ctr_m": ("Clicks ÷ impressions in month M", MONTH_M, "Last day of month M"),
    "position_m": ("Impression-weighted average position in month M", MONTH_M, "Last day of month M"),
    "position_volatility_m": ("How much the daily position moved within month M", MONTH_M, "Last day of month M"),
    "days_with_impressions_m": ("Days in month M with at least one impression", MONTH_M, "Last day of month M"),
    "log_expected_m": ("The baseline's expected clicks in month M (five tiers, client-scaled), log scale", MONTH_M, "Last day of month M"),
    "has_search_volume": ("1 if the keyword's search volume is recorded", KEYWORDS, "Before the page is written"),
    "has_competition": ("1 if the keyword's competition score is recorded", KEYWORDS, "Before the page is written"),
    "has_cpc": ("1 if the keyword's cost per click is recorded", KEYWORDS, "Before the page is written"),
    "has_main_intent": ("1 if the keyword's search intent is recorded", KEYWORDS, "Before the page is written"),
    "log_search_volume": ("Monthly searches for the keyword, log scale; 0 when missing", KEYWORDS, "Before the page is written"),
    "competition": ("Advertiser competition for the keyword, 0 to 1; 0 when missing", KEYWORDS, "Before the page is written"),
    "log_cpc": ("Advertisers' cost per click for the keyword, log scale; 0 when missing", KEYWORDS, "Before the page is written"),
    "keyword_token_count": ("Number of words in the target keyword; never missing", KEYWORDS, "Before the page is written"),
}
NOTES.update({f"content_type={value}": (f"1 if the page is a {value}", KEYWORDS, "When the page is created") for value in CONTENT_TYPES})
NOTES.update({f"intent={value}": (f"1 if the keyword's intent is {value}", KEYWORDS, "Before the page is written") for value in INTENTS})
NOTES["intent=unknown"] = ("1 if the keyword's intent is missing", KEYWORDS, "Before the page is written")
assert set(NOTES) == set(columns), f"Notes and features differ: {sorted(set(NOTES) ^ set(columns))}"
assert not [name for name, note in NOTES.items() if "M+1" in note[1] or "next" in name], "A feature comes from month M+1."

first_pair = next(iter(pairs.values()))
raw_missing = first_pair[["search_volume", "competition", "cpc", "main_intent", "keyword_token_count", "content_type"]].isna().mean()
show_table("Feature notes", pd.DataFrame({
    "Meaning": [note[0] for note in NOTES.values()],
    "Source": [note[1] for note in NOTES.values()],
    "Known by": [note[2] for note in NOTES.values()],
}, index=pd.Index(list(NOTES), name="Feature")))
show_table("Underlying fields missing before any fill, first pair", pd.DataFrame({
    "Missing": raw_missing.map("{:.1%}".format),
    "Handling": ["0 plus has-value flag"] * 3 + ["'unknown' column plus has-value flag", "none needed", "none needed"],
}).rename_axis("Field"))

## 3. The leakage hunt

Three kinds of leak, attacked one at a time, then two checks on the rows themselves and one on privacy.

### 3a. Label-derived features: does any single column give the answer away?

A feature that was built from the target, or from the same numbers, separates positives from negatives almost perfectly on its own.

**Method:** this cell computes each feature's separation (AUC in whichever direction is stronger) against the target on all three pairs, and flags any feature at 0.90 or above.


In [ ]:
def separation(values: pd.Series, target: pd.Series) -> float:
    """Return max(AUC, 1 - AUC): how well one column alone splits positives from negatives, in either direction."""
    auc = roc_auc_score(target, values)
    return float(max(auc, 1 - auc))


scan = pd.DataFrame({name: {feature: separation(vectors[name][feature], rows["target"]) for feature in columns}
                     for name, rows in pairs.items()})
scan["Highest"] = scan.max(axis=1)
scan = scan.sort_values("Highest", ascending=False)
flagged = scan.index[scan["Highest"] >= 0.90].tolist()
show_table("Separation of each feature on its own (0.50 = no information, 1.00 = gives the answer away)",
           scan.map("{:.2f}".format).rename_axis("Feature"))
show("Single-feature scan", {"features at 0.90 or above": ", ".join(flagged) if flagged else "none"})

**Method:** the scan's alarm needs an explanation, not a dismissal. Both flagged features are built only from month M, so they cannot carry month M+1 by construction; the question is why they separate so well. This cell shows the target rate by how many impressions a page had in month M, then recomputes the flagged features' separation among pages with at least 1,000 impressions in month M. If page size is the whole story, the target rate climbs steeply with size, and the separation falls once small pages are left out.


In [ ]:
# Target rate by month-M impressions: a shortfall only counts as real once a page has enough traffic to rule out luck.
BANDS = [0, 100, 1_000, 10_000, 100_000, np.inf]
BAND_NAMES = ["under 100", "100 to 999", "1,000 to 9,999", "10,000 to 99,999", "100,000 or more"]
by_size = {}
for name, rows in pairs.items():
    band = pd.cut(rows["impressions_m"], BANDS, right=False, labels=BAND_NAMES)
    by_size[f"{name}, rows"] = band.value_counts(sort=False).map("{:,}".format)
    by_size[f"{name}, target rate"] = rows["target"].groupby(band, observed=False).mean().map("{:.1%}".format)
show_table("Target rate by impressions in month M", pd.DataFrame(by_size).rename_axis("Impressions in month M"))

# The flagged features (or the two strongest, if none was flagged), on all pages and on pages with 1,000+ impressions.
inspected = flagged or scan.index[:2].tolist()
large = {name: rows["impressions_m"] >= 1_000 for name, rows in pairs.items()}
show_table("Separation of the flagged features, all pages against pages with 1,000 or more impressions", pd.DataFrame({
    f"{name}, {scope}": {feature: f"{separation(vectors[name].loc[mask, feature], rows.loc[mask, 'target']):.2f}" for feature in inspected}
    for name, rows in pairs.items()
    for scope, mask in (("all pages", rows["impressions_m"] >= 0), ("1,000 or more", large[name]))
}).rename_axis("Feature"))

### 3b. Future windows: what if a feature used the wrong month?

Every search feature is built from month M. If one were accidentally built from month M+1, the target's own month, it would know the outcome.

**Method:** this cell rebuilds each search feature from month M+1 and compares its separation with the honest month-M version, on all three pairs.


In [ ]:
# The same six search features, built once from month M and once from month M+1.
def search_features(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Return the search features built from the given month: suffix 'm' for month M, 'next' for month M+1."""
    return pd.DataFrame({
        "log_impressions": np.log1p(rows[f"impressions_{suffix}"]),
        "log_clicks": np.log1p(rows[f"clicks_{suffix}"]),
        "ctr": rows[f"clicks_{suffix}"] / rows[f"impressions_{suffix}"],
        "position": rows[f"position_{suffix}"],
        "position_volatility": rows[f"position_volatility_{suffix}"],
        "days_with_impressions": rows[f"days_with_impressions_{suffix}"],
    })


travel = {}
for name, rows in pairs.items():
    honest, future = search_features(rows, "m"), search_features(rows, "next")
    for feature in honest.columns:
        travel.setdefault(feature, {})[f"{name}, month M"] = separation(honest[feature], rows["target"])
        travel[feature][f"{name}, month M+1"] = separation(future[feature], rows["target"])
show_table("Separation when a feature is built from month M (honest) or month M+1 (leaked)",
           pd.DataFrame(travel).T.map("{:.2f}".format).rename_axis("Feature"))

### 3c. Product flags: do FlyRank's own decisions encode the answer?

The content table records decisions FlyRank made after our months: which pages it later optimized, which it marked as eligible for optimization, and which pages were later deleted or unpublished. If those decisions were driven by the same underperformance the target measures, using them as features would copy FlyRank's decisions instead of finding signal.

**Method:** this cell marks, on each pair, the pages with each product or snapshot flag, and compares the target rate of flagged and unflagged pages. A lift well above 1 means the flag carries information about the outcome that no one could have known in month M.


In [ ]:
def flag_table(rows: pd.DataFrame) -> pd.DataFrame:
    """Return, for each product or snapshot flag, how many rows carry it and the target rate with and without it."""
    flags = {
        "Optimized by FlyRank after the pair": pd.to_datetime(rows["last_optimized_date"]) > rows["pair_end"],
        "Marked eligible for optimization": rows["optimization_eligible_date"].notna(),
        "Deleted by July 2026": rows["is_deleted"].fillna(False).astype(bool),
        "Unpublished by July 2026": ~rows["is_published"].fillna(True).astype(bool),
        "Edited after month M+1": pd.to_datetime(rows["content_updated_date"]) > rows["pair_end"],
    }
    out = {}
    for flag, marked in flags.items():
        with_flag, without = rows.loc[marked, "target"].mean(), rows.loc[~marked, "target"].mean()
        out[flag] = {"Rows flagged": f"{marked.mean():.1%}", "Target rate, flagged": f"{with_flag:.1%}",
                     "Target rate, not flagged": f"{without:.1%}", "Lift": f"{with_flag / without:.2f}x"}
    return pd.DataFrame(out).T


for name, rows in pairs.items():
    show_table(f"Product and snapshot flags against the target, {name}", flag_table(rows).rename_axis("Flag"))

### 3d. The probe model: memorized clients and a planted leak

A model can also cheat without any single leaky column, by recognizing which client a page belongs to. Clients differ in click behaviour, so a model trained and tested on the same clients can score well by remembering them, and that skill vanishes on a new client. The honest test keeps every client's pages in a single fold.

The probe is a small gradient-boosted tree model, used here only as a detector. It is trained five ways on the first pair, and every score is out-of-fold:

1. **Honest features, random split.**
2. **Honest features, client-grouped split.** The honest number.
3. **Honest features plus the client's identity, random split.** How much remembering clients can inflate a score.
4. **Honest features plus the client's identity, client-grouped split.** The same model judged on unseen clients.
5. **Honest features plus a planted leak, client-grouped split.** The planted leak is month M+1's clicks ÷ expected clicks, the target's own ingredient. If the score does not jump towards 1.0, the test harness itself is broken.

**Method:** this cell trains the five variants with five folds each and reports AUC and Precision@50 on the out-of-fold predictions, with the base rate beside them. It takes about a minute.


In [ ]:
def out_of_fold(features: pd.DataFrame, target: pd.Series, folds: list) -> pd.Series:
    """Return a prediction for every row from a probe model that never saw that row during training."""
    predictions = pd.Series(0.0, index=features.index)
    for train, test in folds:
        probe = HistGradientBoostingClassifier(max_iter=200, random_state=0, categorical_features="from_dtype")
        probe.fit(features.iloc[train], target.iloc[train])
        predictions.iloc[test] = probe.predict_proba(features.iloc[test])[:, 1]
    return predictions


def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the share of the top-k rows by score whose target is true."""
    return float(target.loc[score.sort_values(ascending=False).index[:k]].mean())


probe_name = next(iter(pairs))
rows, honest = pairs[probe_name], vectors[probe_name]
target = rows["target"].astype(int)
with_client = honest.assign(client=pd.Categorical(rows["client_hash_id"]).codes).astype({"client": "category"})
with_leak = honest.assign(planted_leak=rows["clicks_next"] / rows["expected_next"])
random_folds = list(KFold(5, shuffle=True, random_state=0).split(honest))
client_folds = list(GroupKFold(5).split(honest, target, rows["client_hash_id"]))

variants = {
    "1. Honest features, random split": (honest, random_folds),
    "2. Honest features, client-grouped split": (honest, client_folds),
    "3. Plus client identity, random split": (with_client, random_folds),
    "4. Plus client identity, client-grouped split": (with_client, client_folds),
    "5. Plus planted leak, client-grouped split": (with_leak, client_folds),
}
probe_results = {}
for variant, (features, folds) in variants.items():
    predictions = out_of_fold(features, target, folds)
    probe_results[variant] = {"AUC": f"{roc_auc_score(target, predictions):.3f}",
                              "Precision@50": f"{precision_at_k(predictions, target, 50):.2f}",
                              "Base rate": f"{target.mean():.1%}"}
show_table(f"Probe model, {probe_name} (out-of-fold scores)", pd.DataFrame(probe_results).T.rename_axis("Variant"))

### 3e. Does choosing the rows leak anything?

The data contract keeps a page only if it has impressions and a valid position in month M+1, and drops pages FlyRank optimized during month M+1. Both rules use month M+1 information. In real use, an editor ranks every page visible in month M without knowing which ones will still be visible next month.

**Method:** this cell rebuilds each pair's population from month M information alone, scores it with the baseline, and follows the baseline's top 50 into month M+1: how many keep their impressions, and how many survive every contract rule.


In [ ]:
def month_m_population(month_m: str) -> pd.DataFrame:
    """Return every page visible in month M with a valid position and a covered client, scored by the baseline."""
    rows = page_months[(page_months["month"] == month_m) & (page_months["position_impressions"] > 0)].copy()
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows["position"] = rows["sum_position"] / rows["position_impressions"]
    rows["expected"] = client_aware_expected(rows, "clicks", "impressions", position_tier(rows["position"]))
    missed = (rows["expected"] - rows["clicks"]).clip(lower=0)
    rows["score"] = missed.where(poisson.cdf(rows["clicks"], rows["expected"]) < 0.01, 0)
    return rows


selection = {}
for (month_m, month_next), (name, rows) in zip(PAIRS, pairs.items(), strict=True):
    population = month_m_population(month_m)
    top = population.nlargest(50, "score")
    visible_next = set(page_months.loc[page_months["month"] == month_next, "content_hash_id"])
    optimized = pd.to_datetime(content["last_optimized_date"]).between(pd.Timestamp(f"{month_m}-01"), rows["pair_end"].iloc[0])
    selection[name] = {
        "Month-M population": f"{len(population):,}",
        "... still visible in M+1": f"{population['content_hash_id'].isin(visible_next).mean():.1%}",
        "Baseline top 50 still visible in M+1": f"{top['content_hash_id'].isin(visible_next).sum()} of 50",
        "Baseline top 50 optimized by FlyRank during the pair": f"{top['content_hash_id'].isin(content.loc[optimized, 'content_hash_id']).sum()} of 50",
        "Baseline top 50 kept by every rule": f"{top['content_hash_id'].isin(rows['content_hash_id']).sum()} of 50",
    }
show_table("Row selection: the month-M view against the contract's rows", pd.DataFrame(selection).T.rename_axis("Pair"))

### 3f. Pages and keywords that repeat

Two kinds of repetition could carry information between training and testing: the same page appearing in several pairs, and the same keyword shared between clients.

**Method:** this cell measures how many pages of each pair also appear in the previous pair, and how many keywords (by their pseudonymous hash) are targeted by more than one client.


In [ ]:
import itertools

# Pages carried over from the previous pair, and keywords shared across clients.
names = list(pairs)
repeat = {names[0]: "first pair"}
for earlier, later in itertools.pairwise(names):
    repeat[later] = f"{pairs[later]['content_hash_id'].isin(pairs[earlier]['content_hash_id']).mean():.1%}"
all_rows = pd.concat(pairs.values(), ignore_index=True)
keyword_clients = all_rows.dropna(subset=["keyword_hash_id"]).groupby("keyword_hash_id")["client_hash_id"].nunique()
show("Repetition", {
    **{f"pages also in the previous pair, {name}": value for name, value in repeat.items()},
    "keywords targeted by more than one client": f"{(keyword_clients > 1).mean():.2%} of {len(keyword_clients):,}",
})


### 3g. Privacy

**Method:** this cell checks that the feature vector holds numbers only, that the identifier columns are pseudonymous hashes and nothing else, that no text column anywhere in the pairs looks like a web address or an email, and that no identifier is a feature.


In [ ]:
# Identifiers must be pseudonymous hashes; no text field may look like a URL or an email address.
HASH = {"client_hash_id": r"^client_[0-9a-f]{16}$", "content_hash_id": r"^content_[0-9a-f]{16}$"}
LOOKS_PRIVATE = r"https?://|www\.|@|\.(?:com|net|org|io)\b"
for column, pattern in HASH.items():
    assert all_rows[column].str.match(pattern).all(), f"{column} holds a value that is not a pseudonymous hash."
text_columns = [column for column in all_rows.columns if pd.api.types.is_string_dtype(all_rows[column]) and column not in HASH]
private_hits = {column: int(all_rows[column].dropna().astype(str).str.contains(LOOKS_PRIVATE, regex=True).sum())
                for column in text_columns}
assert not any(private_hits.values()), f"A text field looks like a URL or an email: {private_hits}"
assert not set(columns) & {"client_hash_id", "content_hash_id", "keyword_hash_id"}, "An identifier is a feature."
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in vectors[probe_name].dtypes), "A feature is not numeric."
show("Privacy checks (all passed)", {
    "feature vector": "numbers only, no identifiers",
    "identifier columns": "pseudonymous hashes only",
    "text columns scanned for URLs or emails": f"{len(text_columns)} columns, 0 matches",
})

## 4. What I excluded and why

Every field that could have been a feature but was refused, with the reason, and a final check that none of them is in the feature vector.


**Method:** this cell lists the refused fields with their reasons and checks the feature vector against the list.


In [ ]:
# Each refused field and the one-line reason; the reasons from the leakage hunt cite the section that tested them.
EXCLUDED = {
    "clicks, impressions, position in month M+1": "The target's own month (section 3b)",
    "expected clicks and p in month M+1": "The target's own ingredients (planted leak, section 3d)",
    "position averaged over months M and M+1": "Overlaps the target month (the data contract's leak test)",
    "last_optimized_date, optimization_eligible_date": "FlyRank decisions, made after month M (section 3c)",
    "is_deleted, is_published": "Page state in July 2026: future information (section 3c)",
    "content_updated_date, word_count, char_count, backlinks, category_count, url_char_count": "Snapshot values that may describe the page after later edits",
    "provider_used, model_used": "FlyRank product decisions",
    "client_hash_id, content_hash_id, keyword_hash_id, url_hash_id": "Pseudonymous identifiers; recognizing clients does not carry to new clients (section 3d)",
    "fact_content_query_90d (all columns)": "Fixed window 2026-04-02 to 2026-06-30, after or inside the target months",
    "GA4 engagement fields": "Real for only a minority of pages; used as a reason code, never as a feature",
    "sessions by channel and AI referrals": "Outside a search click-through question",
    "keyword_char_count, competition_level": "Duplicates of keyword_token_count and competition",
}
refused = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date",
           "word_count", "client_hash_id", "content_hash_id", "keyword_hash_id", "target", "expected_next", "clicks_next",
           "impressions_next", "position_next", "planted_leak", "client"}
assert not refused & set(columns), f"A refused field is in the feature vector: {sorted(refused & set(columns))}"
assert not [column for column in columns if column.endswith("_next")], "A month M+1 field is in the feature vector."
show_table("Refused fields", pd.DataFrame({"Why": list(EXCLUDED.values())}, index=pd.Index(list(EXCLUDED), name="Field")))
show("Final check", {"refused fields in the feature vector": "none", "features carried forward": len(columns)})

## 5. Limitations

- **The probe model is a detector, not a result.** Its scores show where leakage would appear; they say nothing yet about how well a properly tuned model ranks pages.
- **One pair for the probe.** The client-memorization test runs on the first pair only; the single-feature, time-travel, and product-flag tests cover all three.
- **Separation is measured one column at a time.** Two harmless-looking columns could leak together; the probe model's client-grouped score is the guard against that.
- **The row rules use month M+1.** Section 3e measures how much that changes the baseline's view; any final ranking has to be judged against pages visible in month M.
- **Product flags are only partly recorded.** Only the latest optimization date exists, so earlier FlyRank decisions are invisible to these tests.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.